# MongoDB’ye Sıfırdan Giriş ve Python ile CRUD İşlemleri

**Görsel Programlama | 2. sınıf | 3 ders saati**  
**Ön koşul:** Python listeleri, sözlükler, döngüler ve fonksiyonlara aşinalık. **SQL bilgisi gerekmiyor.**  
**Bu dersin kapsamı:** MongoDB Community Server + Compass + PyMongo; PyQt5 entegrasyonu sonraki derste.

> **Öğretim yöntemi:** Açıklama → öğretmenle birlikte çalıştır → Compass'ta sonucu gör → Sıra Sizde → çözüm kontrolü.

### Kazanımlar
1. Bellekteki veri ile kalıcı veri arasındaki farkı açıklar.
2. Database / collection / document / `_id` kavramlarını ayırt eder.
3. Yerel MongoDB sunucusuna bağlanır ve bağlantıyı doğrular.
4. PyMongo ile ekleme, sorgulama, güncelleme ve silme işlemlerini yapar.
5. Basit filtreleme, sıralama, sayma ve hata kontrolleri uygular.
6. Terminal tabanlı küçük bir kitap kayıt sistemi geliştirir.

**Ders akışı (180 dk):** Motivasyon ve kavramlar 25 dk · kurulum/Compass 35 dk · bağlantı ve CRUD 65 dk · laboratuvar 45 dk · değerlendirme 10 dk.

## 1. Neden veritabanı kullanırız?

Bir programda veriyi Python listesi içinde saklamak kolaydır. Ancak liste yalnızca çalışan Python sürecinin belleğinde tutulur. Programı yeniden başlatınca aynı listeyi koddan yeniden oluşturmamız gerekir. Verileri kalıcı tutmak, aramak, güncellemek ve farklı uygulamalardan erişmek için veritabanı kullanabiliriz.

 “1000 öğrencinin kaydını güncellemek için her defasında Python kaynak kodunu değiştirmek ister miyiz?”

In [ ]:
ogrenciler = [
    {"no": 101, "ad": "Ayşe", "yas": 20},
    {"no": 102, "ad": "Mehmet", "yas": 21}
]
for ogrenci in ogrenciler:
    print(ogrenci)
# Kernel yeniden başlatıldığında bu değişken kaybolur.

### JSON, BSON ve MongoDB'nin temel yapısı

- **Database (veritabanı):** İlgili koleksiyonların bulunduğu mantıksal alan. Örn. `okul_db`.
- **Collection (koleksiyon):** Bir grup belge. Örn. `ogrenciler`.
- **Document (belge):** Alan/değer çiftlerinden oluşan kayıt. Python'da sözlüğe benzer.
- **`_id`:** Her belgenin koleksiyon içinde benzersiz kimliği; belirtilmezse sürücü genellikle `ObjectId` oluşturur.
- **JSON:** İnsanların okuyabildiği metin veri gösterimi; **BSON:** MongoDB'nin kullandığı ikili belge biçimi, tarih ve ObjectId gibi ek türleri destekler.

```text
MongoDB Sunucusu
└── okul_db                 (database)
    └── ogrenciler         (collection)
        ├── {"_id": ..., "no": 101, "ad": "Ayşe"}  (document)
        └── {"_id": ..., "no": 102, "ad": "Mehmet"}
```

**Önemli:** MongoDB esnek şemalıdır; bu, alanları rastgele ve kontrolsüz kullanmanın iyi bir fikir olduğu anlamına gelmez.

## 2. Windows'ta kurulum — adım adım

### A. MongoDB Community Server
1. Resmî indirme sayfasını açın: https://www.mongodb.com/try/download/community
2. İşletim sistemi olarak **Windows**, uygun **MSI** paketini seçin. İndirme ekranının görünümü sürüme göre değişebilir.
3. Yükleyiciyi başlatın; standart kurulum adımlarını izleyin.
4. **MongoDB'yi Windows Service olarak kurma** seçeneği sunulursa, laboratuvar bilgisayarında yerel sunucunun kolay başlatılması için seçebilirsiniz.
5. Kurulum bitince Windows **Hizmetler (Services)** ekranında MongoDB servisinin çalıştığını kontrol edin. Gerekirse servisi başlatın.
6. MongoDB Server ile Compass **farklı bileşenlerdir**: Compass kurulmuş olması veritabanı sunucusunun çalıştığı anlamına gelmez.

### B. MongoDB Compass
1. https://www.mongodb.com/try/download/compass adresinden Windows sürümünü indirin.
2. Kurun ve Compass'ı açın.
3. Bağlantı adresine `mongodb://localhost:27017/` yazın, **Connect** düğmesini kullanın.
4. İlk bağlantı başarısızsa önce Server servisinin çalışıp çalışmadığını kontrol edin.

### C. Python ve Jupyter ortamı
PyCharm Terminal / Windows Terminal içinde, notebook'un çalışacağı **aynı Python ortamında**:

```bash
python -m pip install pymongo jupyter
```

Jupyter'da `%pip install pymongo` kullanmak, etkin kernel ortamına kurulum için daha güvenlidir. PyMongo yüklemesi sonrası kernel yeniden başlatılması gerekebilir.

> **Güvenlik:** Bu ders yerel geliştirme içindir. `localhost:27017` adresini internete açmayın. Gerçek uygulamalarda kimlik doğrulama ve erişim kısıtlaması gereklidir. MongoDB Atlas kullanılacaksa bağlantı dizesindeki parolaları notebook'a yazmayın.

In [ ]:
# Notebook kernel'ına PyMongo kurmak için aşağıdaki satırın başındaki # işaretini kaldırın.
# %pip install pymongo

### Compass'ta ilk elle işlem

1. **Create Database** → Database Name: `okul_db`, Collection Name: `ogrenciler`.
2. Koleksiyona girin, **Add Data / Insert Document** ile şu belgeyi ekleyin:

```json
{"no": 100, "ad": "Deniz", "yas": 20, "bolum": "Yazılım Mühendisliği"}
```

3. Documents ekranında **Filter** alanına `{"yas": {"$gte": 20}}` yazıp sorgulayın.
4. Belgeyi açıp bir alanını düzenleyin; değişikliği gözlemleyin.
5. Python ile eklenen kayıtların da aynı koleksiyonda görüneceğini birazdan test edin.

**Dikkat:** Compass arayüzündeki düğme adları sürüme göre küçük farklılıklar gösterebilir.

## 3. Python'dan MongoDB'ye bağlanma

Önce **Server** çalışmalı. PyMongo'nun `MongoClient` nesnesi oluşturulması tek başına sunucuya erişimin başarılı olduğunu garanti etmez; `ping` ile doğrulayacağız.

In [17]:
from pymongo import MongoClient
from pymongo.errors import ServerSelectionTimeoutError

URI = "mongodb://localhost:27017/"
client = MongoClient(URI, serverSelectionTimeoutMS=4000)
try:
    client.admin.command("ping")
    print("✓ MongoDB bağlantısı başarılı")
except ServerSelectionTimeoutError as hata:
    print("✗ MongoDB sunucusuna erişilemiyor. Server servisini ve portu kontrol edin.")
    print(hata)
    raise

db = client["okul_db"]
koleksiyon = db["ogrenciler"]
print("Veritabanı:", db.name, "| Koleksiyon:", koleksiyon.name)

✓ MongoDB bağlantısı başarılı
Veritabanı: okul_db | Koleksiyon: ogrenciler


**Kod açıklaması:** `client` sunucu bağlantısını, `db` veritabanını, `koleksiyon` belge grubunu temsil eder. `client["okul_db"]` ifadesi tek başına fiziksel veritabanı oluşturmak zorunda değildir; ilk yazma işlemiyle oluşturulabilir.

### Tekrar çalıştırma güvenliği

Bu notebook'taki öğretim örneklerini **`ders_kayitlari`** koleksiyonunda, kitap laboratuvarını **`lab_kitaplar`** koleksiyonunda tutacağız. Örnek verileri tekrar eklediğimizde kopyalar oluşmaması için `no` alanında benzersiz indeks kullanacağız. Compass'ta oluşturduğunuz `ogrenciler` koleksiyonunu böylece bozmuyoruz.

**Import sırası:** İlk aşamada yalnızca bağlantı için `MongoClient` ve `ServerSelectionTimeoutError` kullanılır. `DuplicateKeyError` kayıt eklerken, `ASCENDING` ve `pprint` kayıt okurken, `DESCENDING` sıralama örneğinde, `PyMongoError` ise fonksiyonlarla CRUD bölümünde tanıtılır.


In [18]:
ders = db["ders_kayitlari"]
ders.create_index("no", unique=True)
print("Hazır. Mevcut örnek kayıt sayısı:", ders.count_documents({}))

Hazır. Mevcut örnek kayıt sayısı: 0


## 4. CREATE — Kayıt ekleme

### `insert_one(belge)`
Tek bir Python sözlüğünü belge olarak ekler. `inserted_id` eklenen belgenin kimliğidir. Örneği tekrar çalıştırırsanız benzersiz `no` alanı nedeniyle ikinci kez ekleme engellenir.

In [21]:
# Aynı öğrenci numarası tekrar eklenirse oluşan hatayı yakalamak için:
from pymongo.errors import DuplicateKeyError

tek_ogrenci = {"no": 102, "ad": "Ali", "yas": 19, "bolum": "Yazılım"}
try:
    sonuc = ders.insert_one(tek_ogrenci)
    print("Eklenen ID:", sonuc.inserted_id)
except DuplicateKeyError:
    print("101 numaralı öğrenci zaten kayıtlı.")

Eklenen ID: 6aec6ee36abdc3264a59f743


### `insert_many([belge1, belge2, ...])`
Birden fazla belgeyi ekler. Burada notebook'un tekrar çalıştırılabilir olması için örnek kayıtları `update_one(..., upsert=True)` ile hazırlıyoruz; gerçek `insert_many()` kullanımını da ayrı bir örnek koleksiyonunda gösteriyoruz.

In [23]:
ornekler = [
    {"no": 106, "ad": "Mehmet", "yas": 21, "bolum": "Bilgisayar"},
    {"no": 103, "ad": "Zeynep", "yas": 22, "bolum": "Yazılım"},
    {"no": 104, "ad": "Ali", "yas": 19, "bolum": "Yazılım"},
    {"no": 105, "ad": "Elif", "yas": 23, "bolum": "Bilgisayar"},
]
sonuc = ders.insert_many(ornekler)
print("Eklenen ID sayısı:", len(sonuc.inserted_ids))


BulkWriteError: batch op errors occurred, full error: {'writeErrors': [{'index': 0, 'code': 11000, 'errmsg': 'E11000 duplicate key error collection: okul_db.ders_kayitlari index: no_1 dup key: { no: 106 }', 'keyPattern': {'no': 1}, 'keyValue': {'no': 106}, 'op': {'no': 106, 'ad': 'Mehmet', 'yas': 21, 'bolum': 'Bilgisayar', '_id': ObjectId('6aec70746abdc3264a59f748')}}], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 0, 'nMatched': 0, 'nModified': 0, 'nRemoved': 0, 'upserted': []}

In [ ]:
for belge in ornekler:
    ders.update_one({"no": belge["no"]}, {"$setOnInsert": belge}, upsert=True)
print("Örnekler hazır:", ders.count_documents({}))

In [6]:
# insert_many için ayrı bir geçici eğitim koleksiyonu.
coklu = db["coklu_ekleme_demo"]
coklu.delete_many({"ders_etiketi": "demo_1"})  # yalnızca bu demoya ait belgeler
sonuc = coklu.insert_many([
    {"ders_etiketi": "demo_1", "ad": "Kalem"},
    {"ders_etiketi": "demo_1", "ad": "Defter"}
])
print("Eklenen ID sayısı:", len(sonuc.inserted_ids))

Eklenen ID sayısı: 2


### ✍️ Sıra Sizde — Bir kayıt ekleyin

`no=108`, `ad="Ece"`, `yas=20` olan belgeyi `ders` koleksiyonuna ekleyin. Aynı numara zaten varsa kullanıcıya bilgi verin.

<details><summary>✅ Çözümü göster</summary>

```python
try:
    ders.insert_one({"no": 106, "ad": "Ece", "yas": 20})
    print("Eklendi")
except DuplicateKeyError:
    print("Bu numara zaten mevcut")
```

</details>

## 5. READ — Kayıt okuma

- `find()` bir **cursor** döndürür; döngüyle gezilebilir.
- `find_one()` ilk eşleşen belgeyi veya `None` döndürür.
- Filtre sözlüğü `{}` bütün belgeleri eşleştirir.
- Sonuç sırası garanti edilmez; belirli sıra için `sort()` kullanın.

In [44]:
# Okuma sırasında sıralama ve okunabilir çıktı için:
from pymongo import ASCENDING
from pprint import pprint

print("Tüm öğrenciler:")
for belge in ders.find().sort("no", ASCENDING):
    pprint(belge)



Tüm öğrenciler:
{'_id': ObjectId('6aec6e2b6abdc3264a59f741'),
 'ad': 'Ayşe',
 'bolum': 'Yazılım',
 'no': 101,
 'yas': 21}
{'_id': ObjectId('6aec6ee36abdc3264a59f743'),
 'ad': 'Ali',
 'bolum': 'Yazılım',
 'no': 102,
 'yas': 19}
{'_id': ObjectId('6aec700f6abdc3264a59f745'),
 'ad': 'Zeynep',
 'bolum': 'Yazılım',
 'no': 103,
 'yas': 22}
{'_id': ObjectId('6aec700f6abdc3264a59f746'),
 'ad': 'Ali',
 'bolum': 'Yazılım',
 'no': 104,
 'yas': 19}
{'_id': ObjectId('6aec700f6abdc3264a59f747'),
 'ad': 'Elif',
 'bolum': 'Bilgisayar',
 'no': 105,
 'yas': 23}
{'_id': ObjectId('6aec700f6abdc3264a59f744'),
 'ad': 'Mehmet',
 'bolum': 'Bilgisayar',
 'no': 106,
 'yas': 21}


In [25]:
print("Tek kayıt:")
pprint(ders.find_one({"no": 102}))
print("Bulunmayan:", ders.find_one({"no": -1}))

Tek kayıt:
{'_id': ObjectId('6aec6ee36abdc3264a59f743'),
 'ad': 'Ali',
 'bolum': 'Yazılım',
 'no': 102,
 'yas': 19}
Bulunmayan: None


### Projection — yalnızca istediğimiz alanlar

`{"_id": 0, "ad": 1, "yas": 1}` ile `_id` gizlenir, yalnızca ad ve yaş gösterilir. `_id` dışındaki alanlarda dahil etme (1) ve hariç tutmayı (0) aynı projection'da karıştırmayın.

In [26]:
for belge in ders.find({}, {"_id": 0, "ad": 1, "yas": 1}).sort("ad", 1):
    print(belge)

{'ad': 'Ali', 'yas': 19}
{'ad': 'Ali', 'yas': 19}
{'ad': 'Ayşe', 'yas': 20}
{'ad': 'Elif', 'yas': 23}
{'ad': 'Mehmet', 'yas': 21}
{'ad': 'Zeynep', 'yas': 22}


## 6. Filtreleme operatörleri

| Operatör | Anlam | Örnek |
|---|---|---|
| `$eq` | eşit | `{"yas": {"$eq": 20}}` |
| `$ne` | eşit değil | `{"yas": {"$ne": 20}}` |
| `$gt` | büyük | `{"yas": {"$gt": 20}}` |
| `$gte` | büyük/eşit | `{"yas": {"$gte": 20}}` |
| `$lt` | küçük | `{"yas": {"$lt": 20}}` |
| `$lte` | küçük/eşit | `{"yas": {"$lte": 20}}` |
| `$in` | listedekilerden biri | `{"yas": {"$in": [20, 21]}}` |
| `$nin` | listedekilerden hiçbiri | `{"yas": {"$nin": [20, 21]}}` |
| `$exists` | alan var/yok | `{"telefon": {"$exists": True}}` |

**Mantıksal koşullar:** `$and`, `$or`, `$not`, `$nor`. Aynı belgenin iki ayrı alanına verilen koşullar zaten örtük AND ile birleşir.

In [27]:
sorgular = {
    "Yaşı 20 ve üzeri": {"yas": {"$gte": 20}},
    "Yaşı 20 veya 21": {"yas": {"$in": [20, 21]}},
    "Yazılım öğrencileri": {"bolum": "Yazılım"},
    "Yazılım VE 20+": {"$and": [{"bolum": "Yazılım"}, {"yas": {"$gte": 20}}]},
    "19 yaş VEYA Bilgisayar": {"$or": [{"yas": 19}, {"bolum": "Bilgisayar"}]},
}
for baslik, filtre in sorgular.items():
    print("", baslik)
    for belge in ders.find(filtre, {"_id": 0, "no": 1, "ad": 1, "yas": 1}).sort("no", 1):
        print(belge)

 Yaşı 20 ve üzeri
{'no': 101, 'ad': 'Ayşe', 'yas': 20}
{'no': 103, 'ad': 'Zeynep', 'yas': 22}
{'no': 105, 'ad': 'Elif', 'yas': 23}
{'no': 106, 'ad': 'Mehmet', 'yas': 21}
 Yaşı 20 veya 21
{'no': 101, 'ad': 'Ayşe', 'yas': 20}
{'no': 106, 'ad': 'Mehmet', 'yas': 21}
 Yazılım öğrencileri
{'no': 101, 'ad': 'Ayşe', 'yas': 20}
{'no': 102, 'ad': 'Ali', 'yas': 19}
{'no': 103, 'ad': 'Zeynep', 'yas': 22}
{'no': 104, 'ad': 'Ali', 'yas': 19}
 Yazılım VE 20+
{'no': 101, 'ad': 'Ayşe', 'yas': 20}
{'no': 103, 'ad': 'Zeynep', 'yas': 22}
 19 yaş VEYA Bilgisayar
{'no': 102, 'ad': 'Ali', 'yas': 19}
{'no': 104, 'ad': 'Ali', 'yas': 19}
{'no': 105, 'ad': 'Elif', 'yas': 23}
{'no': 106, 'ad': 'Mehmet', 'yas': 21}


### ✍️ Sıra Sizde — Koşullu arama

Yaşı **21’den büyük** öğrencilerin sadece `ad` ve `yas` alanlarını gösterin.

<details><summary>✅ Çözümü göster</summary>

```python
for belge in ders.find({"yas": {"$gt": 21}}, {"_id": 0, "ad": 1, "yas": 1}):
    print(belge)
```

</details>

## 7. `sort()`, `limit()`, `skip()`, `count_documents()`, `distinct()`

- `sort("yas", 1)` artan, `sort("yas", -1)` azalan sıralar.
- `limit(3)` en fazla üç belge getirir.
- `skip(2)` ilk iki sonucu atlar; sayfalama için kararlı sıralamayla birlikte kullanılır.
- `count_documents(filtre)` eşleşen belge sayısını verir.
- `distinct("bolum")` ilgili alanın farklı değerlerini döndürür.

In [29]:
# Azalan sıralama için DESCENDING sabitini kullanabiliriz.
from pymongo import DESCENDING

print("Yaşa göre en büyük 3:")
for belge in ders.find({}, {"_id": 0, "ad": 1, "yas": 1}).sort([("yas", DESCENDING), ("ad", ASCENDING)]).limit(3):
    print(belge)


Yaşa göre en büyük 3:
{'ad': 'Elif', 'yas': 23}
{'ad': 'Zeynep', 'yas': 22}
{'ad': 'Mehmet', 'yas': 21}


In [34]:
print("Toplam:", ders.count_documents({}))


Toplam: 6


In [35]:
print("20+ sayısı:", ders.count_documents({"yas": {"$gte": 20}}))


20+ sayısı: 4


In [36]:
print("Bölümler:", ders.distinct("bolum"))


Bölümler: ['Bilgisayar', 'Yazılım']


In [38]:
print("İkinci sayfa (2 kayıt):")
for belge in ders.find({}, {"_id": 0, "no": 1, "ad": 1}).sort("no", 1).skip(4):
    print(belge)

İkinci sayfa (2 kayıt):
{'no': 105, 'ad': 'Elif'}
{'no': 106, 'ad': 'Mehmet'}


## 8. UPDATE — Kayıt güncelleme

`update_one(filtre, değişiklik)` bir belgeyi; `update_many(...)` bütün eşleşen belgeleri günceller. Sonuçta `matched_count` eşleşen, `modified_count` gerçekten değişen belge sayısıdır.

| Operatör | İşlev |
|---|---|
| `$set` | alanı oluştur/değiştir |
| `$inc` | sayısal değeri artır/azalt |
| `$unset` | alanı kaldır |
| `$rename` | alan adını değiştir |
| `$push` | dizi alanına eleman ekle |
| `$addToSet` | dizide yoksa ekle |
| `$pull` | diziden eşleşen elemanı çıkar |

**Uyarı:** `replace_one()` tüm belgeyi ( `_id` hariç) değiştirir; `update_one()` ise belirtilen alanları günceller.

In [42]:
print(ders.find_one({"no": 101}, {"_id": 0}))

{'no': 101, 'ad': 'Ayşe', 'yas': 21, 'bolum': 'Yazılım'}


In [40]:
sonuc = ders.update_one({"no": 101}, {"$set": {"yas": 21}})
print("Eşleşen:", sonuc.matched_count, "Değişen:", sonuc.modified_count)



Eşleşen: 1 Değişen: 1


In [45]:

# $inc her çalıştırmada sayıyı artırır; bunu ayrı demo belgesinde göstereceğiz.
ders.update_one({"no": 999}, {"$setOnInsert": {"no": 999, "ad": "Sayaç demosu", "puan": 0}}, upsert=True)


UpdateResult({'n': 1, 'upserted': ObjectId('6aec7ab6f46711fce0f92a84'), 'nModified': 0, 'ok': 1.0, 'updatedExisting': False}, acknowledged=True)

In [46]:
ders.update_one({"no": 999}, {"$inc": {"puan": 1}})
print("Sayaç:", ders.find_one({"no": 999}, {"_id": 0, "puan": 1}))

Sayaç: {'puan': 1}


In [47]:
# update_many: yalnızca bu örneklerdeki kayıtlar için güvenli ve tekrar çalıştırılabilir.
sonuc = ders.update_many({"no": {"$in": [101, 102, 103, 104, 105]}}, {"$set": {"ders": "Görsel Programlama"}})
print("Eşleşen:", sonuc.matched_count, "Değişen:", sonuc.modified_count)



Eşleşen: 5 Değişen: 5


In [48]:
# $unset: geçici bir alan ekle ve kaldır.
ders.update_one({"no": 102}, {"$set": {"gecici": "silinsin"}})


UpdateResult({'n': 1, 'nModified': 1, 'ok': 1.0, 'updatedExisting': True}, acknowledged=True)

In [49]:
ders.update_one({"no": 102}, {"$unset": {"gecici": ""}})
pprint(ders.find_one({"no": 102}, {"_id": 0}))

{'ad': 'Ali',
 'bolum': 'Yazılım',
 'ders': 'Görsel Programlama',
 'no': 102,
 'yas': 19}


In [50]:
pprint(ders.find_one({"no": 103}, {"_id": 0}))

{'ad': 'Zeynep',
 'bolum': 'Yazılım',
 'ders': 'Görsel Programlama',
 'no': 103,
 'yas': 22}


In [51]:
ders.update_one({"no": 103}, {"$set": {"bolum": "Elektrik Mühendisliği"}})
print(ders.find_one({"no": 103}, {"_id": 0}))

{'no': 103, 'ad': 'Zeynep', 'yas': 22, 'bolum': 'Elektrik Mühendisliği', 'ders': 'Görsel Programlama'}


### ✍️ Sıra Sizde — Güncelleme

`no=103` öğrencisinin `bolum` alanını `Yazılım Mühendisliği` olarak değiştirin. Sonucu tekrar sorgulayın.

<details><summary>✅ Çözümü göster</summary>

```python
ders.update_one({"no": 103}, {"$set": {"bolum": "Yazılım Mühendisliği"}})
print(ders.find_one({"no": 103}, {"_id": 0}))
```

</details>

## 9. DELETE — Kayıt silme

- `delete_one(filtre)` en fazla bir belgeyi siler.
- `delete_many(filtre)` tüm eşleşen belgeleri siler.
- **`delete_many({})` bütün belgeleri siler. Bu derste ana koleksiyonda çalıştırmayın.**
- Silmeden önce aynı filtreyi `find()` ile test etmek iyi alışkanlıktır.

Aşağıdaki örnekte yalnızca `silme_demo` koleksiyonuna dokunuyoruz.

In [52]:
silme_demo = db["silme_demo"]
silme_demo.delete_many({"kaynak": "notebook"})

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)

In [59]:

silme_demo.insert_many([
    {"kaynak": "notebook", "kod": 1, "ad": "A"},
    {"kaynak": "notebook", "kod": 2, "ad": "B"},
    {"kaynak": "notebook", "kod": 3, "ad": "C"},
])
print("Silmeden önce:", list(silme_demo.find({"kaynak": "notebook"}, {"_id": 0})))



Silmeden önce: [{'kaynak': 'notebook', 'kod': 1, 'ad': 'A'}, {'kaynak': 'notebook', 'kod': 1, 'ad': 'A'}, {'kaynak': 'notebook', 'kod': 1, 'ad': 'A'}, {'kaynak': 'notebook', 'kod': 1, 'ad': 'A'}, {'kaynak': 'notebook', 'kod': 2, 'ad': 'B'}, {'kaynak': 'notebook', 'kod': 3, 'ad': 'C'}]


In [56]:
sonuc = silme_demo.delete_one({"kaynak": "notebook", "kod": 1})
print("Tek silme:", sonuc.deleted_count)


Tek silme: 1


In [58]:
sonuc = silme_demo.delete_many({"kaynak": "notebook", "kod": {"$gte": 2}})
print("Çoklu silme:", sonuc.deleted_count)

Çoklu silme: 8


### ✍️ Sıra Sizde — Güvenli silme

`no=106` olan öğrenci varsa önce bilgilerini gösterin, ardından silin; yoksa uyarı verin.

<details><summary>✅ Çözümü göster</summary>

```python
kayit = ders.find_one({"no": 106})
if kayit is None:
    print("Öğrenci bulunamadı")
else:
    print("Silinecek:", kayit)
    sonuc = ders.delete_one({"no": 106})
    print("Silinen:", sonuc.deleted_count)
```

</details>

## 10. Ek yararlı komutlar ve iyi uygulamalar

| Komut | Açıklama |
|---|---|
| `db.list_collection_names()` | koleksiyon adlarını listeler |
| `client.list_database_names()` | görülebilen veritabanlarını listeler |
| `collection.create_index("no", unique=True)` | benzersiz indeks oluşturur |
| `collection.find_one_and_update(...)` | güncellenen belgeyi alma senaryolarında kullanılır |
| `collection.aggregate([...])` | toplulaştırma/raporlama (ileri konu) |
| `collection.drop()` | koleksiyonu tamamen kaldırır (**tehlikeli**) |
| `client.close()` | istemci kaynaklarını kapatır |

### İndeks neden önemli?

Öğrenci numarası gibi tekil olması gereken alanlara benzersiz indeks koymak, uygulama kodunda kontrol yapılmasa bile tekrar kayıt oluşmasını önlemeye yardımcı olur. Mevcut mükerrer değerler varsa benzersiz indeks oluşturma işlemi başarısız olabilir.

### Hata yönetimi

`PyMongoError` bağlantı ve veritabanı işlemlerindeki çeşitli hataların üst sınıfıdır. `DuplicateKeyError` benzersiz indeks ihlali, `ServerSelectionTimeoutError` sunucuya erişim sorunu gibi durumlar için kullanılabilir.

In [60]:
print("Koleksiyonlar:", db.list_collection_names())
print("Veritabanları:", client.list_database_names())
print("İndeksler:", list(ders.index_information().keys()))

Koleksiyonlar: ['coklu_ekleme_demo', 'ders_kayitlari', 'silme_demo']
Veritabanları: ['admin', 'config', 'dbname', 'dpu_sinav_planlama', 'dpu_sinav_programi_ders', 'dpu_staj', 'kutuphane_db', 'local', 'mydatabase', 'okul_db', 'pyqt_mongodb_example', 'school_database', 'student_db', 'student_info', 'student_management', 'task_management', 'test_database', 'user_database', 'users_db']
İndeksler: ['_id_', 'no_1']


## 11. fonksiyonlarla öğrenci kaydı

Aşağıdaki fonksiyonlar PyQt5'e geçiş için altyapıdır. Kullanıcı arayüzünden bağımsız veritabanı işlevleri yazıyoruz.

In [61]:
# Veritabanı işlemlerindeki genel hataları yakalamak için:
from pymongo.errors import PyMongoError

def ogrenci_ekle(no, ad, yas):
    if not isinstance(no, int) or not isinstance(yas, int) or not ad.strip():
        return "Geçersiz bilgi"
    try:
        ders.insert_one({"no": no, "ad": ad.strip(), "yas": yas})
        return "Kayıt eklendi"
    except DuplicateKeyError:
        return "Bu öğrenci numarası zaten kayıtlı"
    except PyMongoError as hata:
        return f"Veritabanı hatası: {hata}"

def ogrenci_ara(no):
    return ders.find_one({"no": no}, {"_id": 0})

def ogrenci_guncelle(no, yeni_ad):
    if not yeni_ad.strip():
        return "Ad boş olamaz"
    sonuc = ders.update_one({"no": no}, {"$set": {"ad": yeni_ad.strip()}})
    return "Bulunamadı" if sonuc.matched_count == 0 else "İşlem tamamlandı"

def ogrenci_sil(no):
    sonuc = ders.delete_one({"no": no})
    return "Silindi" if sonuc.deleted_count else "Bulunamadı"

print(ogrenci_ekle(201, "Merve", 20))
print(ogrenci_ara(201))
print(ogrenci_guncelle(201, "Merve Nur"))
print(ogrenci_ara(201))
# Silme işlemini öğrencilere yaptırın: print(ogrenci_sil(201))

Kayıt eklendi
{'no': 201, 'ad': 'Merve', 'yas': 20}
İşlem tamamlandı
{'no': 201, 'ad': 'Merve Nur', 'yas': 20}


## 12. Laboratuvar — Kitap Kayıt Sistemi (45 dakika)

**Amaç:** Öğrenci, Python terminalinden yönetilen bir kitap sistemi geliştirsin. 

### Zorunlu özellikler
1. `kutuphane_db` veritabanı ve `lab_kitaplar` koleksiyonu.
2. Kitap bilgileri: `kitap_no` (benzersiz), `ad`, `yazar`, `yil`, `tur`.
3. Menü: 1 Ekle · 2 Listele · 3 Ara · 4 Güncelle · 5 Sil · 0 Çıkış.
4. Yanlış menü seçimi, boş kitap adı ve sayıya çevrilemeyen girişleri kontrol et.
5. Silme öncesi kullanıcıdan onay iste.
6. Compass üzerinden en az bir kaydın gerçekten saklandığını göster.

**Teslim:** `kitap_kayit.py`, kısa açıklama ve Compass ekran görüntüsü. **Ek görev:** Aynı `kitap_no` ile tekrar eklemeyi engelle.

### Başlangıç iskeleti — öğrenciler tamamlayacak

In [ ]:
# Laboratuvar için başlangıç: fonksiyon gövdelerini öğrenciler yazacak.
kitaplar = client["kutuphane_db"]["lab_kitaplar"]
kitaplar.create_index("kitap_no", unique=True)

def kitap_ekle():
    # TODO: input() ile alanları al, kontrol et, insert_one kullan
    pass

def kitap_listele():
    # TODO: find() ve sort() kullan
    pass

def kitap_ara():
    # TODO: find_one() kullan
    pass

def kitap_guncelle():
    # TODO: update_one() ve $set kullan
    pass

def kitap_sil():
    # TODO: önce bul, onay al, delete_one() kullan
    pass

# Menü döngüsünü de öğrenciler tamamlayacak.

<details><summary>🔐 Öğretmen için: Tam laboratuvar çözümünü göster (öğrenciye başta açmayın)</summary>

Aşağıdaki hücre **tam çözümdür**. Öğrencilere önce yukarıdaki iskeleti verin. Tam çözümde giriş doğrulaması, benzersiz indeks ve silme onayı bulunur.

</details>

In [ ]:
# ÖĞRETMEN ÇÖZÜMÜ — İstenirse ayrı bir .py dosyasına kopyalanıp çalıştırılır.
# Bu hücre yalnızca fonksiyonları tanımlar; menü otomatik başlamaz.
from pymongo.errors import DuplicateKeyError, PyMongoError

lab = client["kutuphane_db"]["lab_kitaplar"]
lab.create_index("kitap_no", unique=True)

def tam_kitap_ekle():
    try:
        no = int(input("Kitap no: "))
        ad = input("Kitap adı: ").strip()
        yazar = input("Yazar: ").strip()
        yil = int(input("Yayın yılı: "))
        tur = input("Tür: ").strip()
        if not ad or not yazar or not tur or yil < 0:
            print("Alanları geçerli şekilde doldurun.")
            return
        lab.insert_one({"kitap_no": no, "ad": ad, "yazar": yazar, "yil": yil, "tur": tur})
        print("✓ Kitap eklendi")
    except ValueError:
        print("Kitap no ve yıl tam sayı olmalıdır")
    except DuplicateKeyError:
        print("Bu kitap numarası zaten kayıtlı")
    except PyMongoError as hata:
        print("Veritabanı hatası:", hata)

def tam_kitap_listele():
    for kitap in lab.find({}, {"_id": 0}).sort("kitap_no", 1):
        print(kitap)

def tam_kitap_ara():
    try:
        no = int(input("Aranacak kitap no: "))
        print(lab.find_one({"kitap_no": no}, {"_id": 0}) or "Bulunamadı")
    except ValueError:
        print("Geçerli bir numara girin")

def tam_kitap_guncelle():
    try:
        no = int(input("Güncellenecek kitap no: "))
        yeni_ad = input("Yeni kitap adı: ").strip()
        if not yeni_ad:
            print("Kitap adı boş olamaz")
            return
        sonuc = lab.update_one({"kitap_no": no}, {"$set": {"ad": yeni_ad}})
        print("Güncellendi" if sonuc.matched_count else "Bulunamadı")
    except ValueError:
        print("Geçerli bir numara girin")

def tam_kitap_sil():
    try:
        no = int(input("Silinecek kitap no: "))
        kitap = lab.find_one({"kitap_no": no})
        if kitap is None:
            print("Bulunamadı")
            return
        print("Silinecek:", kitap["ad"])
        if input("Onaylıyor musunuz? (e/h): ").strip().lower() != "e":
            print("İptal edildi")
            return
        sonuc = lab.delete_one({"_id": kitap["_id"]})
        print("Silindi" if sonuc.deleted_count else "Bulunamadı")
    except ValueError:
        print("Geçerli bir numara girin")

def tam_menu():
    islemler = {"1": tam_kitap_ekle, "2": tam_kitap_listele,
                "3": tam_kitap_ara, "4": tam_kitap_guncelle, "5": tam_kitap_sil}
    while True:
        print("\n1-Ekle 2-Listele 3-Ara 4-Güncelle 5-Sil 0-Çıkış")
        secim = input("Seçim: ").strip()
        if secim == "0":
            print("Görüşmek üzere")
            break
        if secim in islemler:
            try:
                islemler[secim]()
            except PyMongoError as hata:
                print("Veritabanı hatası:", hata)
        else:
            print("Geçersiz seçim")

# Terminalde başlatmak için yorum işaretini kaldırın:
# tam_menu()

## 13. Hata bulma etkinlikleri

**Önce öğrenciler yanıtlasın; ardından çözümü açın.**

### 🔎 Yanlış veri tipi

```python
koleksiyon.insert_one([{"ad":"Ali"},{"ad":"Ayşe"}])
```

<details><summary>Çözümü göster</summary>

```python
koleksiyon.insert_many([{"ad":"Ali"},{"ad":"Ayşe"}])
```

`insert_one` tek sözlük bekler; liste için `insert_many` gerekir.

</details>

### 🔎 Operatör yazımı

```python
ders.find({"yas":{"gt":20}})
```

<details><summary>Çözümü göster</summary>

```python
ders.find({"yas":{"$gt":20}})
```

Sorgu operatörleri `$` ile başlar.

</details>

### 🔎 Güncelleme sözdizimi

```python
ders.update_one({"no":101},{"yas":22})
```

<details><summary>Çözümü göster</summary>

```python
ders.update_one({"no":101},{"$set":{"yas":22}})
```

Güncelleme belgesinde `$set` gibi operatör kullanılmalıdır.

</details>

### 🔎 Boş filtre tehlikesi

```python
ders.delete_many({})
```

<details><summary>Çözümü göster</summary>

```python
ders.delete_one({"no":101})
```

Boş filtre tüm belgeleri eşleştirir; silmede hedef belirlenmelidir.

</details>

### 🔎 Cursor görüntüleme

```python
print(ders.find())
```

<details><summary>Çözümü göster</summary>

```python
print(list(ders.find().limit(5)))
```

`find()` cursor döndürür; belgeleri görmek için cursor üzerinde dolaşın.

</details>

## 14. Ders sonu mini sınav (cevaplar açılır)

1. Database, collection ve document arasındaki ilişki nedir?
2. Python sözlüğü MongoDB'de neye karşılık gelir?
3. `find()` ile `find_one()` farkı nedir?
4. `{"yas": {"$gte": 20}}` filtresi ne demektir?
5. `update_one()` içindeki `$set` neden kullanılır?
6. `delete_many({})` neden risklidir?
7. `inserted_id` ve `modified_count` neyi ifade eder?
8. Compass ile MongoDB Server aynı şey midir?

<details><summary>📘 Cevap anahtarını göster</summary>

1. Database koleksiyonları, koleksiyon belgeleri içerir.
2. Sözlük benzeri alan/değer yapısı bir belgeyi temsil eder.
3. `find` cursor, `find_one` bir belge veya `None` döndürür.
4. Yaşı 20 veya daha fazla olanları eşleştirir.
5. İstenen alanı diğer alanları koruyarak günceller/ekler.
6. Boş filtre tüm kayıtları eşleştirir ve siler.
7. Eklenen belgenin kimliği ve gerçekten değişen belge sayısıdır.
8. Hayır; Server veritabanı hizmeti, Compass grafiksel istemcidir.

</details>

## 15. Sık karşılaşılan sorunlar

| Belirti | Olası neden | Çözüm |
|---|---|---|
| `ModuleNotFoundError: pymongo` | Yanlış Python ortamına kurulum | Notebook'ta `%pip install pymongo`, ardından kernel restart |
| `ServerSelectionTimeoutError` | Server kapalı/yanlış port | Windows Services ve `localhost:27017` kontrolü |
| `DuplicateKeyError` | Benzersiz alan tekrar kullanıldı | Farklı `no` girin; benzersiz indeks mantığını açıklayın |
| `find_one()` → `None` | Filtreye uyan kayıt yok | Alan adı ve veri tipini kontrol edin (`101` ≠ `"101"`) |
| Compass bağlanıyor, Python bağlanmıyor | URI veya çalışma ortamı farklı | İki tarafta aynı bağlantı adresini kullanın |
| Aynı kayıtlar çoğalıyor | Ekleme hücresi tekrar çalıştırılıyor | Benzersiz indeks / `upsert` kullanın |

**Kapanış:** MongoDB verileri kalıcı saklar; Python bu verileri yönetir. **Sonraki derste** Qt Designer ile hazırlanan formu PyMongo fonksiyonlarına bağlayıp `QTableWidget` üzerinde kayıtları göstereceğiz.

### Resmî kaynaklar
- MongoDB Community Server: https://www.mongodb.com/try/download/community
- MongoDB Compass: https://www.mongodb.com/try/download/compass
- PyMongo kurulum ve kullanım: https://www.mongodb.com/docs/languages/python/pymongo-driver/current/
- CRUD işlemleri: https://www.mongodb.com/docs/manual/crud/
- Sorgu operatörleri: https://www.mongodb.com/docs/manual/reference/operator/query/

> **Son not:** Dersin bitiminde isteğe bağlı olarak `client.close()` çalıştırın. Notebook'ta sonraki hücreler bağlantıyı kullanacaksa kapatmayı en sona bırakın.

In [ ]:
# BÜTÜN İŞLEMLER BİTTİKTEN SONRA çalıştırın:
# client.close()
print("Ders tamamlandı. Bağlantıyı kapatmak için yukarıdaki satırı etkinleştirin.")

In [16]:
from pymongo import MongoClient

# MongoDB sunucusuna bağlan
client = MongoClient("mongodb://localhost:27017/")

# Veritabanını seç
db = client["okul_db"]

# Koleksiyonu seç
ogrenciler = db["ogrenciler"]

# Koleksiyonu tamamen sil
ogrenciler.drop()

print("ogrenciler koleksiyonu silindi.")

client.close()

ogrenciler koleksiyonu silindi.


In [9]:
# Geçici koleksiyon oluştur
silme_test = db["koleksiyon_silme_demo"]

# Veri ekle
silme_test.insert_one({"no": 1, "ad": "Deneme"})

# Koleksiyonu tamamen sil
silme_test.drop()